# Résoudre `3_free` avec OR-Tools
On reprend la démarche des notebooks 08 à 11 sur le troisième jeu de données : 5 000 modèles et 20 000 datasets. Comme dans le notebook 11, on écrit le problème en équations et on le confie au solveur SCIP. Mais on commence par regarder **ce qui change** par rapport à `2_medium`, parce que le modèle mathématique doit suivre les données.

**Prérequis** : `datasets/3_free.json`, `test_solution.py`, `pandas`, `ortools`.

**Durée d'exécution** : moins d'une minute.

## 1. Charger l'instance et le repère

In [1]:
from pathlib import Path
from collections import defaultdict
import sys, json, time, bisect
import pandas as pd
from ortools.linear_solver import pywraplp
pd.set_option("display.width", 150)

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))
from test_solution import getSolutionScore

DATASET = ROOT_DIR / "datasets" / "3_free.json"
DATASET_TXT = DATASET.read_text()
instance = json.loads(DATASET_TXT)
energy_cap = instance["energyCap"]
print("commentaire de l'instance :", instance["comment"])

VALUE  = {m["id"]: m["value"] for m in instance["models"]}
ENERGY = {m["id"]: m["energyCost"] for m in instance["models"]}
NEEDS  = {m["id"]: [(t, b["lowerBound"], b["upperBound"]) for t, b in m["dataRequirements"].items()] for m in instance["models"]}
SIZE   = {d["id"]: d["size"] for d in instance["datasets"]}
TYPE   = {d["id"]: d["dataType"] for d in instance["datasets"]}
COPY   = {d["id"]: d["isCopyrighted"] for d in instance["datasets"]}
IMPOSSIBLE = {m for m, needs in NEEDS.items() if any(lo > hi for _, lo, hi in needs)}
LOWER = {m: {t: lo for t, lo, hi in needs if lo <= hi} for m, needs in NEEDS.items()}
MONO = {m: needs[0][0] for m, needs in NEEDS.items() if len(needs) == 1 and m not in IMPOSSIBLE}

WINNER = 523_758
print(f"{len(VALUE)} modèles, {len(SIZE)} datasets | gagnant du concours : {WINNER:,}")

commentaire de l'instance : Everything is free!
5000 modèles, 20000 datasets | gagnant du concours : 523,758


## 2. Ce qui change par rapport à `2_medium`

In [2]:
supply = pd.Series({t: sum(SIZE[d] for d in SIZE if TYPE[d] == t) for t in "ntic"})
free = pd.Series({t: sum(SIZE[d] for d in SIZE if TYPE[d] == t and COPY[d] == 0) for t in "ntic"})
demand = pd.Series({t: sum(LOWER[m].get(t, 0) for m in VALUE) for t in "ntic"})
print(f"datasets sous droits : {sum(COPY.values())} sur {len(COPY)}")
print(f"énergie de tous les modèles : {sum(ENERGY.values()) / energy_cap:.2f} × le plafond")
print(f"modèles à énergie quasi nulle : {sum(e <= 10 for e in ENERGY.values())} | modèles impossibles : {len(IMPOSSIBLE)}")

# Environ un modèle sur huit tient dans le plafond : quelle part des données ces modèles demanderaient-ils ?
share = energy_cap / sum(ENERGY.values())
pd.DataFrame({"offre": supply, "dont_libre": free, "demande_si_tout": demand,
              "demande_probable": (demand * share).round(), "offre_sur_demande_probable": (supply / (demand * share)).round(1)})

datasets sous droits : 0 sur 20000
énergie de tous les modèles : 7.95 × le plafond
modèles à énergie quasi nulle : 293 | modèles impossibles : 94


,offre,dont_libre,demande_si_tout,demande_probable,offre_sur_demande_probable
n,24727061,24727061,59341893,7460151.0,3.3
t,24952470,24952470,60213913,7569777.0,3.3
i,25957507,25957507,59301535,7455078.0,3.5
c,24644719,24644719,58602327,7367177.0,3.3


**Lecture**
- **Aucun dataset n'est sous droits.** Le malus de demi-valeur ne peut plus venir que des sources. La contrainte qui nous a coûté le plus cher sur `2_medium`, la rareté des données libres, **disparaît**.
- L'énergie de tous les modèles fait **8 fois le plafond** : il faut toujours choisir environ un modèle sur huit.
- Les modèles qu'on pourra entraîner demanderont environ un tiers de l'offre de chaque type. **Les données ne sont plus une contrainte.**

Le problème devient un **sac à dos pur sur l'énergie**, plus les sources pour les 94 modèles impossibles. On peut donc simplifier le modèle du notebook 11.

## 3. Le modèle mathématique, simplifié
Les variables $y_i$ (demi-valeur) disparaissent : avec des données toutes libres, un modèle entraîné avec des datasets est toujours à pleine valeur. Il reste :
- $x_i$ : le modèle $i$ est entraîné avec des datasets (pleine valeur) ;
- $z_m$ : le modèle impossible $m$ est entraîné grâce à des sources (demi-valeur) ;
- $u_{s,m,t}$ : le modèle $s$ (à un seul besoin, de type $t$) sert de source au besoin impossible $t$ de $m$.

$$\max \sum_i v_i\,x_i + \sum_m \lfloor v_m/2 \rfloor\,z_m$$

sous les contraintes : un seul rôle par modèle ($x_i + \sum u_{i,\cdot} \le 1$), une source par besoin impossible ($\sum_s u_{s,m,t} = z_m$), le plafond d'énergie (sources comprises) et l'offre totale de données par type.

C'est un **sac à dos 0-1** de 5 000 objets, avec un petit problème d'appariement en plus. La programmation dynamique, la méthode exacte classique du sac à dos, est impossible ici : elle demanderait un tableau indexé par l'énergie, soit $3 \times 10^{14}$ cases. Un solveur MIP, lui, s'appuie sur la relaxation linéaire et n'est pas gêné par la taille des coefficients.

In [3]:
def solve_model(seconds=120, relative_gap=None):
    """Construit et résout le modèle. relative_gap : tolérance d'arrêt de SCIP (None = réglage par défaut)."""
    t0 = time.time()
    S = pywraplp.Solver.CreateSolver("SCIP")
    x = {m: S.BoolVar(f"x{m}") for m in VALUE if m not in IMPOSSIBLE}
    z = {m: S.BoolVar(f"z{m}") for m in IMPOSSIBLE}
    u = {(s, m, t): S.BoolVar(f"u{s}_{m}_{t}")
         for m in IMPOSSIBLE for t, lo, hi in NEEDS[m] if lo > hi
         for s, ts in MONO.items() if ts == t}
    as_source, sources_of = defaultdict(list), defaultdict(list)
    for (s, m, t), var in u.items():
        as_source[s].append(var)
        sources_of[(m, t)].append(var)

    for m in x:
        S.Add(x[m] + sum(as_source[m]) <= 1)                              # un seul rôle
    for (m, t), vars_ in sources_of.items():
        S.Add(sum(vars_) == z[m])                                          # une source par besoin impossible
    S.Add(sum(ENERGY[m] * x[m] for m in x) + sum(ENERGY[m] * z[m] for m in z)
          + sum(ENERGY[s] * var for (s, m, t), var in u.items()) <= energy_cap)       # énergie
    for t in "ntic":                                                        # données (jamais saturées ici)
        S.Add(sum(LOWER[m][t] * x[m] for m in x if t in LOWER[m]) + sum(LOWER[m][t] * z[m] for m in z if t in LOWER[m])
              + sum(LOWER[s][t] * var for (s, m, tt), var in u.items() if tt == t) <= supply[t])
    S.Maximize(sum(VALUE[m] * x[m] for m in x) + sum((VALUE[m] // 2) * z[m] for m in z))

    S.SetTimeLimit(seconds * 1000)
    params = pywraplp.MPSolverParameters()
    if relative_gap is not None:
        params.SetDoubleParam(params.RELATIVE_MIP_GAP, relative_gap)
    status = S.Solve(params)
    val = lambda v: round(v.solution_value())
    sol = dict(full=[m for m in x if val(x[m])], impossible=[m for m in z if val(z[m])],
               sources=[(s, m, t) for (s, m, t), var in u.items() if val(var)])
    obj, bound = S.Objective().Value(), S.Objective().BestBound()
    info = dict(statut="OPTIMAL" if status == pywraplp.Solver.OPTIMAL else "FEASIBLE", objectif=round(obj),
                borne=round(bound, 1), écart_prouvé=round(bound - obj, 1), secondes=round(time.time() - t0),
                modèles=len(sol["full"]), impossibles=len(sol["impossible"]), sources=len(sol["sources"]))
    return sol, info

## 4. Résoudre : attention au mot « optimal »
On lance SCIP deux fois : avec ses réglages par défaut, puis en exigeant un écart nul.

In [4]:
sol_default, info_default = solve_model()
sol_exact, info_exact = solve_model(relative_gap=0.0)
pd.DataFrame([info_default, info_exact], index=["SCIP par défaut", "SCIP, écart exigé = 0"])

,statut,objectif,borne,écart_prouvé,secondes,modèles,impossibles,sources
SCIP par défaut,OPTIMAL,523747,523759.3,12.3,18,1323,8,8
"SCIP, écart exigé = 0",OPTIMAL,523758,523758.0,0.0,23,1322,9,9


**Lecture**
- Avec ses réglages par défaut, SCIP s'arrête et annonce « OPTIMAL » alors qu'il reste un écart entre la solution et la borne. C'est voulu : par défaut, SCIP s'arrête dès que l'écart relatif passe sous **0,01 %**. Ici, 0,01 % de 523 000 représente jusqu'à 52 points, ce qui suffit à perdre la première place.
- En exigeant un écart nul, SCIP prouve l'optimum en quelques secondes : **la solution et la borne coïncident**.
- À retenir : « optimal » chez un solveur veut toujours dire « optimal **à la tolérance près** ». Il faut connaître cette tolérance.

## 5. Attribuer les datasets
On reprend `Pool`, `fill` (avec la recherche de paire du notebook 11) et `pack`, en simplifiant : il n'y a plus de distinction libre / sous droits.

Une différence importante avec le notebook 11 : les modèles ordinaires, les **sources** et les **cibles** passent dans **une seule file d'attente**. Dans une première version, je servais les sources et les cibles à la fin. Une cible qui demandait entre 1 et 6 unités de type `t` ne trouvait plus de dataset assez petit, et la solution devenait **invalide** (score 0). Une cible mal servie ne fait pas que perdre des points : elle invalide toute la soumission. Si une cible ou une de ses sources ne peut pas être servie, on abandonne donc toute la grappe.

In [5]:
class Pool:
    """Réserve de datasets d'un type, triée par taille."""
    def __init__(self, ids):
        self.ids = sorted(ids, key=SIZE.get)
        self.sizes = [SIZE[d] for d in self.ids]
    def take(self, i):
        self.sizes.pop(i)
        return self.ids.pop(i)
    def put(self, d):
        i = bisect.bisect_left(self.sizes, SIZE[d])
        self.sizes.insert(i, SIZE[d]); self.ids.insert(i, d)

def find_pair(sizes, a, b):
    i, j, best = 0, bisect.bisect_right(sizes, b) - 1, None
    while i < j:
        s = sizes[i] + sizes[j]
        if s > b:
            j -= 1
        elif s < a:
            i += 1
        else:
            if best is None or s < best[0]:
                best = (s, i, j)
            j -= 1
    return best

def fill(pool, lower, upper):
    taken, total = [], 0
    while total < lower:
        a, b = lower - total, upper - total
        i = bisect.bisect_left(pool.sizes, a)
        if i < len(pool.sizes) and pool.sizes[i] <= b:
            total += pool.sizes[i]; taken.append(pool.take(i)); continue
        pair = find_pair(pool.sizes, a, b) if pool.sizes and a <= 2 * pool.sizes[-1] else None
        if pair:
            _, i, j = pair
            total += pool.sizes[i] + pool.sizes[j]
            taken.append(pool.take(j)); taken.append(pool.take(i)); continue
        j = bisect.bisect_right(pool.sizes, b) - 1
        if j < 0:
            break
        total += pool.sizes[j]; taken.append(pool.take(j))
    if lower <= total <= upper:
        return taken
    for d in taken:
        pool.put(d)
    return None

def give(pools, model, needs):
    got = []
    for t, lo, hi in needs:
        res = fill(pools[t], lo, hi)
        if res is None:
            for d in got:
                pools[TYPE[d]].put(d)
            return None
        got += res
    return [[int(d), int(model)] for d in got]

def pack(sol, order_key):
    """Attribue les datasets. Modèles ordinaires, sources et cibles passent dans UNE seule file, triée par order_key.
    Rend (plan, modèles qu'on n'a pas pu servir)."""
    jobs = [(m, NEEDS[m]) for m in sol["full"]]
    jobs += [(s, NEEDS[s]) for s, m, t in sol["sources"]]
    jobs += [(m, [n for n in NEEDS[m] if n[1] <= n[2]]) for m in sol["impossible"]]   # cibles : besoins remplissables
    pools = {t: Pool([d for d in SIZE if TYPE[d] == t]) for t in "ntic"}
    data, failed = {}, []
    for m, needs in sorted(jobs, key=lambda job: order_key(job[1])):
        got = give(pools, m, needs)
        if got is None:
            failed.append(m)
        else:
            data[m] = got
    # Si une cible ou l'une de ses sources a échoué, on abandonne toute la grappe (cible + sources)
    sources = [(s, m) for s, m, t in sol["sources"]]
    broken = {m for s, m in sources if s in failed or m in failed}
    for s, m in sources:
        if m in broken:
            data.pop(s, None); data.pop(m, None)
    plan = {"dataMappings": [p for m, got in data.items() for p in got],
            "modelMappings": [[int(s), int(m)] for s, m in sources if m not in broken]}
    return plan, failed

results = {}
def evaluate(name, plan):
    score, valid, message = getSolutionScore(json.dumps(plan), DATASET_TXT, show_details=False)
    results[name] = dict(score=score, valide=valid, écart_au_gagnant=score - WINNER)
    print(f"{name} : score = {score:,} | valide = {valid} {message} | écart au gagnant = {score - WINNER:+,}")
    return score

### Premier essai : l'ordre du notebook 11
Les modèles aux plus petits besoins d'abord.

In [6]:
by_small_needs = lambda needs: sum(lo for t, lo, hi in needs)
plan_a, failed_a = pack(sol_exact, by_small_needs)
print("modèles non servis :", failed_a, "->", [NEEDS[m] for m in failed_a])
evaluate("A : solveur + ordre « petits besoins »", plan_a)

modèles non servis : [2107, 3601] -> [[('i', 1, 1995), ('n', 1, 9)], [('t', 1, 6), ('c', 1, 55), ('n', 1, -634), ('i', 1, 1693)]]
A : solveur + ordre « petits besoins » : score = 523,651 | valide = True  | écart au gagnant = -107


523651

**Lecture** : deux modèles ne sont pas servis, et la solution perd 107 points. Regardez leurs besoins :
- le modèle 2107 demande entre **1 et 9** unités de type `n`. Seuls **trois datasets** `n` sont assez petits (tailles 4, 5 et 8) ;
- le modèle 3601 est une **cible** de source : il demande entre **1 et 6** unités de type `t`. Comme il échoue, toute sa grappe (cible et source) est abandonnée.

Dans les deux cas, d'autres modèles servis avant eux, dont l'intervalle acceptait aussi de petites tailles, ont pris les petits datasets : `fill` choisit toujours le plus petit dataset qui termine le remplissage.

Le modèle agrégé ne voit pas cette **concurrence pour les petits datasets** : pour lui, il y a des millions d'unités de données disponibles. C'est la même famille d'erreur que dans le notebook 11 : **l'agrégation oublie la granularité**.

### Deuxième essai : les plus contraints d'abord
Règle classique de l'ordonnancement et des problèmes de coloriage : **servir d'abord ceux qui ont le moins de choix**. Un besoin dont la borne haute est petite ne peut accepter que de petits datasets, qui sont rares. On trie donc par borne haute minimale, puis par taille des besoins.

In [7]:
most_constrained = lambda needs: (min((hi for t, lo, hi in needs), default=0), sum(lo for t, lo, hi in needs))
plan_b, failed_b = pack(sol_exact, most_constrained)
print("modèles non servis :", failed_b)
evaluate("B : solveur + ordre « plus contraints d'abord »", plan_b)

modèles non servis : []
B : solveur + ordre « plus contraints d'abord » : score = 523,758 | valide = True  | écart au gagnant = +0


523758

## 6. Et sans solveur ?
Pour mesurer ce qu'apporte le solveur, on reprend la règle gloutonne la plus simple du notebook 09 (plan B) : la meilleure densité `valeur / énergie` d'abord, sans aucune source.

In [8]:
def greedy_density():
    pools = {t: Pool([d for d in SIZE if TYPE[d] == t]) for t in "ntic"}
    energy, mappings = 0, []
    for m in sorted((m for m in VALUE if m not in IMPOSSIBLE), key=lambda m: -VALUE[m] / ENERGY[m]):
        if energy + ENERGY[m] <= energy_cap:
            got = give(pools, m, NEEDS[m])
            if got is not None:
                mappings += got; energy += ENERGY[m]
    return {"dataMappings": mappings, "modelMappings": []}

evaluate("glouton par densité (sans solveur)", greedy_density())

glouton par densité (sans solveur) : score = 523,484 | valide = True  | écart au gagnant = -274


523484

## 7. Comparaison et vérification

In [9]:
summary = pd.DataFrame(results).T
summary.loc["gagnant du concours", ["score", "écart_au_gagnant"]] = [WINNER, 0]
summary.loc["optimum prouvé du modèle", ["score", "écart_au_gagnant"]] = [info_exact["borne"], info_exact["borne"] - WINNER]
display(summary.sort_values("score"))

best_name = max((k for k in results if k.startswith(("A", "B"))), key=lambda k: results[k]["score"])
best_plan = plan_a if best_name.startswith("A") else plan_b
score, valid, message = getSolutionScore(json.dumps(best_plan), DATASET_TXT, show_details=True)
print("meilleur :", best_name, "| score officiel =", score, "| valide =", valid, message)

,score,valide,écart_au_gagnant
glouton par densité (sans solveur),523484,True,-274
A : solveur + ordre « petits besoins »,523651,True,-107
B : solveur + ordre « plus contraints d'abord »,523758,True,0
gagnant du concours,523758,NaN,0
optimum prouvé du modèle,523758.0,NaN,0.0



Total value: 523758
Energy used: 321312929993252 / 321313262728600 (100.00%)
Number of used models: 1340 / 5000 (26.80%)
  Number of devalued models: 9 (0.67%)
  Number of models used as data source: 9 (0.67%)
Number of used datasets: 6867 / 20000 (34.34%)
meilleur : B : solveur + ordre « plus contraints d'abord » | score officiel = 523758 | valide = True 


**Lecture**
- Le plan B atteint **exactement le score du gagnant**, et ce score est l'**optimum prouvé** du modèle. Puisque toute solution réelle respecte les contraintes du modèle, on ne peut pas faire mieux (à une hypothèse près : on ne prévoit des sources que pour les besoins impossibles, et sur cette instance une source pour un besoin remplissable ne ferait que coûter de l'énergie).
- Le glouton par densité est déjà très bon, mais il manque des points que seul un arbitrage global peut trouver : le dernier morceau d'énergie, et les appariements source-cible.

## 8. Écrire le fichier de solution

In [10]:
out_dir = ROOT_DIR / "solutions"; out_dir.mkdir(exist_ok=True)
out = out_dir / "3_free_main.json"
if valid:
    out.write_text(json.dumps(best_plan))
    print("écrit :", out)

écrit : /home/claude/proj/solutions/3_free_main.json


## À retenir
1. **Le modèle suit les données.** Toutes les données sont libres : les variables de demi-valeur disparaissent, et le problème devient un sac à dos sur l'énergie. Avant de modéliser, il faut toujours regarder quelles contraintes sont vraiment actives.
2. **« Optimal » veut dire « optimal à la tolérance près ».** Par défaut, SCIP s'arrête à 0,01 % de l'optimum. Ici, cela représente assez de points pour perdre la première place.
3. **L'agrégation oublie la granularité.** Dans le notebook 11, c'était le découpage des dernières données libres. Ici, ce sont quelques petits datasets que tout le monde veut.
4. **Servir d'abord les plus contraints.** Cette règle, qui vient de l'ordonnancement, règle le problème d'attribution. Sur `2_medium`, elle n'est pas toujours meilleure que l'ordre « petits besoins » : le plus sûr est d'essayer les deux et de garder le meilleur.
5. **Un solveur exact sur un problème bien modélisé, c'est imbattable.** On atteint l'optimum prouvé en moins d'une minute, sans recuit ni réparation.